<a href="https://colab.research.google.com/github/potato-master369/MamboAlgo/blob/main/Copy_of_FAIR_Module1_GettingStarted_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FAIR Module 1: Getting Started — Tools, Data & the ML Mindset

*Some cells are exercises or puzzles for you to complete. Each exercise has one optional hint; full solutions are at the very end of this notebook.*

### What you'll be able to do by the end of this notebook
- Operate this cloud notebook environment and use Python's `random` module to generate data
- Explain what "AI", "ML", "Deep Learning", "Generative AI" and "AGI" actually mean, and how they relate
- Distinguish supervised, unsupervised, self-supervised, and reinforcement learning, and classification from regression
- Represent data as vectors and matrices, and calculate L1/L2 distance and cosine similarity between two vectors
- Explain what a random seed does, and use it to generate **reproducible** random numbers

### How this notebook works
- Run every cell **in order**, top to bottom — later cells depend on earlier ones.
- 📚 **Read on the course page** — at a few points, this notebook pauses and asks you to go read a specific lesson on the course page. The full explanation lives there, not here — come back once you've read it.
- 🔍 **Notice** prompts ask you to look at an output and think before reading on. Each has a collapsed 💡 explanation underneath to check your thinking.
- 🐍 **Python note** boxes explain any code syntax you may not have seen before — this notebook assumes very little prior Python experience.
- 📘 **[Python Reference Guide](https://drive.google.com/file/d/1UgS2oBPQ6Mr07yR7IH83dHQt8LuZLhHq/view?usp=sharing)** — every piece of Python this notebook uses, all in one place: symbols that are easy to mix up (like `=` and `==`), how indentation works, and the code patterns you'll use again and again. Open it in another tab now and keep it there while you work.
- Most of the coding comes in **four steps**, each one building on the one before:
  - ▶️ **Run & Read** — working code for you to run and read. Every line has a comment saying what it does.
  - 🔧 **Tweak It** — change a value or two in working code. Predict what will happen *before* you run it, then check. Where you see 📝, note down the result: the course page asks for it at the end of this module.
  - 🧩 **Put It in Order** — the lines of a working function, shuffled, plus one line that doesn't belong. Put them back in order, and the cell checks your answer for you.
  - 🎯 **Try It Yourself** — write similar code on your own.
- 🌟 **Stretch Challenge** exercises are optional extras. 🎯 and 🌟 exercises each have one optional 🔒 **Hint** immediately after them — a nudge in the right direction, collapsed by default, click to expand.
- ✍️ **Explain** asks you to write a short answer in your own words.
- ✅ **Check** at the end runs automatically and gives you a Module 1 completion code for the course page.
- 🔑 **Solutions & Explanations**, right at the end of this notebook, has the full worked answer to every exercise and puzzle, each with a plain-language explanation of how it works. If you get stuck, or just want to check your work once you're done, that's where to look — there's no need to guess.

This notebook supplies its own data and setup. Work from the top and complete each required exercise before moving on. An unfinished exercise can stop execution; see the recovery instructions below.

## Step 0 — Setup

The required activities use Python's standard-library modules, which come with Python. The optional preview uses third-party packages already available in Colab. Before you run anything, open the 📘 [Python Reference Guide](https://drive.google.com/file/d/1UgS2oBPQ6Mr07yR7IH83dHQt8LuZLhHq/view?usp=sharing) in another tab if you haven't already.

> ⚠️ **When you run your first cell,** Colab may show a message: *“Warning: This notebook was not authored by Google.”* That's expected. Colab shows it for any notebook written by someone other than Google, and this one was written by the FAIR programme team. As long as you opened this notebook from the link on your course page, choose **Run anyway**. (Outside FAIR, only do this for notebooks from people or organisations you trust: a notebook can run any code it contains.)

Now run the cell below.

> 🐍 **Python note:** the statement `import` makes a module's tools available to this code. A module is a collection of reusable code; importing it is different from installing a package. `random` gives you tools for generating random numbers, `math` gives you mathematical functions like square roots, `hashlib` is used once at the very end of this notebook, and `Counter` (from the `collections` toolkit) counts how often things appear in a list. You don't need to memorise any of this now — each one is explained again the first time it's actually used.

In [ ]:
# Run this cell first.
import random
import math
import hashlib
from collections import Counter

print("Environment ready. Let's get started.")

Environment ready. Let's get started.


Now run the next cell as well. It sets up the checker for the 🧩 **Put It in Order** puzzles later in this notebook. It's long, and you don't need to read it — just run it and look for the message `Puzzle checker ready.`

In [ ]:

def _require_vector(v):
    """Check the input contract: a non-empty sequence of numbers."""
    if len(v) == 0:
        raise ValueError("Use a non-empty vector (a list with at least one number).")


def _require_pair(v1, v2):
    """Matching positions must describe matching quantities."""
    _require_vector(v1)
    _require_vector(v2)
    if len(v1) != len(v2):
        raise ValueError("Use vectors with the same number of positions; zip would otherwise ignore extra values.")


def _expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return True
    return False


def _same_numbers(actual, expected):
    return (isinstance(actual, (list, tuple)) and len(actual) == len(expected)
            and all(isinstance(x, (int, float)) and not isinstance(x, bool)
                    and math.isclose(x, y, rel_tol=1e-12, abs_tol=1e-12)
                    for x, y in zip(actual, expected)))


def _check_measurements(function):
    # Independent reference generators avoid disturbing the notebook's random state.
    for n, low, high, seed in [(5, 0.0, 10.0, 123), (4, 2.0, 3.0, 8),
                                (6, -4.0, -1.0, 0), (0, 1.0, 2.0, 7)]:
        reference = random.Random(seed)
        expected = [reference.uniform(low, high) for _ in range(n)]
        first = function(n, low, high, seed=seed)
        second = function(n, low, high, seed=seed)
        assert _same_numbers(first, expected), (
            f"For n={n}, bounds=({low}, {high}), seed={seed}, return {n} uniform samples. "
            "Set the seed once before the loop, append each draw, and return the list. "
            "Repeated constants or resetting the seed inside the loop do not match the reference.")
        assert _same_numbers(second, expected), "Repeating the same seeded call should reproduce the list."
    # No seed supplied: continue the existing state; do not reset it.
    reference = random.Random(31)
    expected = [reference.uniform(-2.0, 4.0) for _ in range(6)]
    state = random.getstate()
    try:
        random.seed(31)
        actual = function(3, -2.0, 4.0) + function(3, -2.0, 4.0)
        assert _same_numbers(actual, expected), "With seed=None, continue the generator's existing state."
    finally:
        random.setstate(state)
    return True




# Puzzle support: run this cell; you do not need to study its implementation.

def _test_counts(fn):
    result = fn([4, 1, 4, 6, 4, 1])
    if result is None:
        return ("Your function gave back nothing (None). Is the return line included, "
                "and is it the very last line, after the loop has finished?")
    if result != {4: 3, 1: 2, 6: 1}:
        return (f"For the rolls [4, 1, 4, 6, 4, 1] your function gave {result}, "
                "but it should give {4: 3, 1: 2, 6: 1}. Look at which lines sit inside the loop.")
    for values in [[], [2, 2, 2], [6, 1, 6, 1, 2]]:
        if fn(values) != dict(Counter(values)):
            return "Try another list: create counts once, update each roll, and return after the loop."
    return None

def _test_dot(fn):
    result = fn([3, 1, 4], [5, 1, 2])
    if result is None:
        return ("Your function gave back nothing (None). Is the return line included, "
                "and is it the very last line, after the loop has finished?")
    if result != 24:
        return (f"For A = [3, 1, 4] and B = [5, 1, 2] your function gave {result}, but the dot product "
                "is 15 + 1 + 8 = 24. Check which lines sit inside the loop, and where total starts.")
    for a, b in [([-2, 0], [1, 4]), ([2], [3]), ([1, 2], [0, 0])]:
        if fn(a, b) != sum(x*y for x, y in zip(a, b)):
            return "Multiply matching positions and add every product to the running total."
    return None

def _test_dice(fn):
    first = fn(5, 20, seed=3)
    second = fn(5, 20, seed=3)
    if first is None:
        return ("Your function gave back nothing (None). Is the return line included, "
                "and is it the very last line, after the loop has finished?")
    if len(first) != 5:
        return (f"Asked for 5 rolls, but got {len(first)}: {first}. "
                "Check which lines sit inside the loop, and that return comes after it.")
    if any(r < 1 or r > 20 for r in first):
        return f"Some rolls fall outside 1 to 20: {first}."
    if first != second:
        return ("Two runs with seed=3 gave different rolls. The seed only affects numbers made "
                "AFTER it is set, so it has to be set before the loop starts making them.")
    for n, sides, seed in [(7, 6, 0), (0, 8, 9)]:
        rng = random.Random(seed)
        expected = [rng.randint(1, sides) for _ in range(n)]
        if fn(n, sides, seed=seed) != expected:
            return "Set the seed once before drawing; append one roll on each iteration."
    return None

PUZZLES = {'counts': {'name': 'count_each',
            'lines': {'A': '            counts[r] = counts[r] + 1',
                      'B': 'def count_each(rolls):',
                      'C': '    return counts',
                      'D': '        else:',
                      'E': '    counts = {}',
                      'F': '        counts = {}',
                      'G': '    for r in rolls:',
                      'H': '            counts[r] = 1',
                      'I': '        if r in counts:'},
            'stray': 'F',
            'stray_msg': 'Line F would reset `counts` to an empty dictionary '
                         'every time the loop goes round, so each roll would '
                         "wipe out everything counted before it. It's the line "
                         "that doesn't belong: take it out and run the cell "
                         'again.'},
 'dot': {'name': 'dot_product',
         'lines': {'A': '    for a, b in zip(v1, v2):',
                   'B': '        total = a * b',
                   'C': '    return total',
                   'D': 'def dot_product(v1, v2):',
                   'E': '        total = total + a * b',
                   'F': '    total = 0'},
         'stray': 'B',
         'stray_msg': 'Line B *replaces* `total` with just the current product '
                      'each time round the loop, instead of adding to it, so '
                      "only the very last product would be kept. It's the line "
                      "that doesn't belong: take it out and run the cell "
                      'again.'},
 'dice': {'name': 'roll_any_die',
          'lines': {'A': '    for _ in range(n_rolls):',
                    'B': '        random.seed(seed)',
                    'C': '    return rolls',
                    'D': '        rolls = random.randint(1, sides)',
                    'E': 'def roll_any_die(n_rolls, sides, seed=None):',
                    'F': '    rolls = []',
                    'G': '        rolls.append(random.randint(1, sides))',
                    'H': '    if seed is not None:'},
          'stray': 'D',
          'stray_msg': 'Line D would *replace* the whole list with a single '
                       "number, instead of adding one more roll to it. It's "
                       "the line that doesn't belong: take it out and run the "
                       'cell again.'}}

PUZZLES['counts']['test'] = _test_counts

PUZZLES['dot']['test'] = _test_dot

PUZZLES['dice']['test'] = _test_dice

def check_puzzle(puzzle_id, my_order):
    """Put your chosen lines together in your order, run them, and test the result."""
    puzzle = PUZZLES[puzzle_id]
    globals().pop(puzzle["name"], None)  # do not reuse a previous answer after a failed attempt
    lines = puzzle["lines"]
    if not my_order:
        print('✏️  my_order is still empty. Type your letters between the square brackets, '
              'e.g. ["C", "A", "B"], then run this cell again.')
        return
    chosen = [str(x).strip().upper() for x in my_order]
    unknown = [x for x in chosen if x not in lines]
    if unknown:
        print(f"❌ {unknown} aren't letters from this puzzle. Use only: {sorted(lines)}")
        return
    repeats = sorted({x for x in chosen if chosen.count(x) > 1})
    if repeats:
        print(f"❌ You used {repeats} more than once. Each line can only be used once.")
        return
    if puzzle["stray"] in chosen:
        print("❌ " + puzzle["stray_msg"])
        return
    code = "\n".join(lines[x] for x in chosen)
    print("Your lines, in your order:\n")
    print(code)
    print()
    missing = [x for x in sorted(lines) if x != puzzle["stray"] and x not in chosen]
    if missing:
        print(f"❌ You haven't used {missing} yet. Every line is needed except the one that doesn't belong.")
        return
    workspace = dict(globals())
    try:
        exec(code, workspace)
    except SyntaxError:
        print("❌ Python can't read the lines in this order. Usually this means an indented line has "
              "nothing above it to belong to (for example, a line from inside a loop placed before "
              "the loop starts), or the def line isn't first.")
        return
    try:
        problem = puzzle["test"](workspace[puzzle["name"]])
    except NameError as err:
        print(f"❌ The function crashed: {err}. This usually means a line uses a name before the "
              "line that creates it has run. Look at the order of your lines.")
        return
    except Exception as err:
        print(f"❌ The function crashed with this error: {type(err).__name__}: {err}")
        return
    if problem:
        print("❌ " + problem)
        return
    globals()[puzzle["name"]] = workspace[puzzle["name"]]
    print(f"✅ Correct! Your lines make a working {puzzle['name']}() function, "
          "and it's now ready to use in the rest of this notebook.")

print("Puzzle checker ready.")

Puzzle checker ready.


> 🛟 **If your session resets:** a disconnected browser does not always mean the runtime has been deleted. Reconnect first. If variables are missing, run both Setup cells and then your completed required cells in order up to your current activity (or use **Runtime → Run before** for that cell). Restore any missing puzzle answers by rerunning their cells. An unfinished exercise may stop execution; complete it before continuing. Skip unfinished optional challenges. Your saved code and text remain in the notebook, but runtime variables must be recreated. Do not use Run all as a shortcut through an unfinished notebook.

### Try It: Installing a Package

This notebook doesn't need anything extra installed, but later modules might, so it's worth seeing how. In Colab, `%pip install` followed by a package name installs it into your current session. Run the cell below: because scikit-learn is already installed in Colab, it will simply report that the requirement is already satisfied — which is exactly what you want to see.

> 🐍 **Python note:** the `%` at the start marks a *notebook command* rather than ordinary Python code. Installed packages last only for the current session, so a notebook that needs one reinstalls it each time it runs from the top.

In [ ]:
%pip install scikit-learn

## Step 1 — Encounter: Rolling the Dice

Let's write a small function that simulates rolling a six-sided die several times.

> 🐍 **Python note:** `def roll_dice(n_rolls, seed=None):` *defines* a function called `roll_dice` — a named, reusable block of code. `n_rolls` and `seed` are its **parameters**: names that receive values when the function is called. The supplied values, such as `10` or `42`, are called **arguments**. `seed=None` means "if nobody says otherwise, use `None`" — you'll find out exactly what `seed` does later in this notebook. The line `return [random.randint(1, 6) for _ in range(n_rolls)]` uses a **list comprehension**: a compact way of writing "build a list by doing something, once for each item in a range." It's exactly equivalent to writing a `for` loop that appends to an empty list one item at a time — just shorter. The underscore `_` is a throwaway variable name Python programmers use when they don't actually need the loop counter itself, only the fact that it's looping. A few cells down, you'll see this same function written out the long way, with a plain `for` loop.

In [ ]:
def roll_dice(n_rolls, seed=None):
    """Generate a list of n_rolls dice rolls, each between 1 and 6."""
    if seed is not None:
        random.seed(seed)
    return [random.randint(1, 6) for _ in range(n_rolls)]

# Try it out — run this cell a couple of times
rolls_a = roll_dice(10)
rolls_b = roll_dice(10)
print("First run: ", rolls_a)
print("Second run:", rolls_b)

First run:  [4, 1, 3, 4, 3, 2, 6, 6, 1, 4]
Second run: [3, 5, 3, 3, 5, 3, 1, 2, 5, 1]


### 🔍 Notice

Are `rolls_a` and `rolls_b` the same? Run the cell above a few more times. What do you get?

This isn't just a quirk of dice. Machine learning uses randomness like this all the time: to shuffle data, to split it, and to choose starting values. If every run gives a different answer, how could you tell whether a change you made actually helped?

Hold that thought. You'll keep using this exact function for the rest of the notebook — sometimes with an extra `seed=...` argument you haven't been told the meaning of yet. Near the end of this notebook, right before your work gets checked automatically, you'll find out exactly what that argument does and why it matters.

<details>
<summary>💡 <b>What's going on here?</b></summary>

They are almost certainly different. Python's pseudorandom generator keeps an internal state: information that determines its next output. Each draw advances that state. In our function, `seed=None` skips resetting it, so the second call continues where the first left off. It does not obtain a new starting point from the system on every call. Different sequences can occasionally match by chance.

That's a real answer, but not the *useful* one yet — the Reproducibility & Random Seeds lesson later in this notebook covers how to control this deliberately, which is the part that actually matters.

</details>

### ▶️ Run & Read: The Same Function, the Long Way

`roll_dice()` builds its list in one compact line. Here is exactly the same function written the long way, with a plain `for` loop and a comment on every line. Run it, then read it line by line. Given the same seed, both versions produce identical rolls. (What the seed actually does is still coming, near the end of this notebook.)

> 🐍 **Python note:** `rolls.append(x)` adds `x` to the end of the list `rolls`. `random.randint(1, 6)` gives one random whole number from 1 to 6, including both 1 and 6.

In [ ]:
def roll_dice_long(n_rolls, seed=None):
    """Exactly the same as roll_dice(), written with a for loop instead of a list comprehension."""
    if seed is not None:                      # was a seed given?
        random.seed(seed)                     # if so, hand it to random.seed() (more on this later)
    rolls = []                                # start with an empty list
    for _ in range(n_rolls):                  # repeat n_rolls times
        rolls.append(random.randint(1, 6))    # add one random whole number from 1 to 6
    return rolls                              # hand the finished list back

print("Compact version:", roll_dice(5, seed=3))
print("Long version:   ", roll_dice_long(5, seed=3))

Compact version: [2, 5, 5, 2, 3]
Long version:    [2, 5, 5, 2, 3]


### 🔧 Tweak It: Change the Dice

The cell below is another copy of the same function, called `roll_tweak()`, for you to experiment with. `roll_dice()` itself stays untouched, so nothing later in the notebook breaks. Make each change **one at a time**, and before you run the cell, predict what will happen.

1. Change the `10` in `roll_tweak(10, seed=2026)` to `5`. *Predict:* how many rolls will print?
2. Change the `6` in `random.randint(1, 6)` to `20`, to turn it into a 20-sided die. *Predict:* what is the biggest number that *could* now appear?
3. Change `seed=2026` to `seed=None`, and run the cell twice. *Predict:* will the two runs match? Then change it back to `seed=2026`.
4. 📝 **Note this down:** with 5 rolls, a 20-sided die, and `seed=2026`, what **sum** does the cell print? You'll be asked for it on the course page at the end of this module.

> 🐍 **Python note:** `len(rolls)`, `max(rolls)` and `sum(rolls)` give a list's length, its biggest value, and its total.

In [ ]:
def roll_tweak(n_rolls, seed=None):
    if seed is not None:
        random.seed(seed)
    rolls = []
    for _ in range(n_rolls):
        rolls.append(random.randint(1, 20))    # <- change 2: the 6
    return rolls

rolls = roll_tweak(10, seed=None)             # <- change 1: the 10       change 3: the seed
print("Rolls:   ", rolls)
print("How many:", len(rolls))
print("Biggest: ", max(rolls))
print("Sum:     ", sum(rolls))

Rolls:    [15, 5, 19, 10, 1, 12, 12, 15, 14, 3]
How many: 10
Biggest:  19
Sum:      106


<details>
<summary>💡 <b>What's going on here?</b></summary>

1. **Five.** `n_rolls` is how many times the loop repeats, and each time round adds exactly one roll to the list.
2. **20.** `random.randint(1, 20)` can give any whole number from 1 to 20. In only a handful of rolls, though, you may never actually see a 20. What *could* appear isn't the same as what *will* appear.
3. **Almost certainly not.** With `seed=None`, this function continues the generator's existing sequence. With `seed=2026`, it resets the generator before drawing, so the same calls in the same environment reproduce the same rolls. Why that happens is the last lesson in this module.

Notice what you just did: you changed a program and predicted its behaviour before running it. That habit, *predict, then check*, is how programmers (and researchers) find out whether they really understand the code in front of them.

</details>

## Step 2 — 📚 Read on the Course Page: What Is Machine Learning?

You've now got the cloud notebook environment working, and a small function that generates data for you to play with. Before going any further, it's worth stepping back and asking two questions: why would we want a computer to **learn**, instead of following rules we write for it? And what does learning actually mean?

Head to the [**What Is Machine Learning?** lesson](https://fair.aisingapore.org/lesson/what-is-machine-learning-3/) now — it starts with a problem that's hard to solve with hand-written rules, then covers the AI/ML/DL/GenAI/AGI family of terms (with a diagram), the four learning paradigms, classification vs. regression, and the general ML workflow (with a diagram). You'll use this vocabulary for the rest of FAIR.

Come back here once you've read it.

### 🎯 Try It Yourself: Classify These Scenarios

For each scenario below, decide: (a) which learning paradigm it is — supervised, unsupervised, self-supervised, or reinforcement — and (b) if supervised, whether it's classification or regression (write `"n/a"` if it isn't supervised).

1. Predicting a house's sale price from its size, location, and age, using thousands of past sales with known prices.
2. Grouping news articles into topic clusters, with no pre-existing topic labels.
3. A robot vacuum learning to clean more area per battery charge through repeated trial runs, rewarded for coverage.
4. Sorting emails into “spam” or “not spam” using a large set of emails that have already been labelled by humans.

> 🐍 **Python note:** the curly braces `{ }` below create a **dictionary** — a collection of `key: value` pairs, a bit like a two-column table. `my_answers[1]` looks up the entry for key `1`, which is itself another small dictionary with `"paradigm"` and `"task"` as its own keys. This is called a **nested** dictionary — a dictionary inside a dictionary.

Fill in your answers below, commit to them, then run the next cell to reveal the answer key and compare.

In [ ]:
# 🔧 TODO: fill in your answers as strings, e.g. "supervised" and "classification" (or "n/a")
my_answers = {
    1: {"paradigm": "supervised", "task": "regression"},
    2: {"paradigm": "unsupervised", "task": "n/a"},
    3: {"paradigm": "reinforcement", "task": "n/a"},
    4: {"paradigm": "supervised", "task": "?"},
}

<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

Ask two questions for each scenario: **Are there labelled examples with known correct answers?** (if yes → supervised; if targets are constructed from the original data itself → self-supervised; if training finds structure without target labels → unsupervised; if it's learning from trial-and-error rewards → reinforcement.) Then, only for supervised cases: **is the answer a category (classification) or a number (regression)?**

</details>

Run the cell below *after* you've committed to your answers above — it reveals the answer key and compares.

In [ ]:
# --- Answer key — run this only after answering above ---
answer_key = {
    1: {"paradigm": "supervised", "task": "regression"},
    2: {"paradigm": "unsupervised", "task": "n/a"},
    3: {"paradigm": "reinforcement", "task": "n/a"},
    4: {"paradigm": "supervised", "task": "classification"},
}

for i in range(1, 5):
    mine, correct = my_answers[i], answer_key[i]
    match = "✅" if mine == correct else "❌"
    print(f"{match} Scenario {i}: you said {mine}, answer key says {correct}")

❌ Scenario 1: you said {'paradigm': '?', 'task': '?'}, answer key says {'paradigm': 'supervised', 'task': 'regression'}
❌ Scenario 2: you said {'paradigm': '?', 'task': '?'}, answer key says {'paradigm': 'unsupervised', 'task': 'n/a'}
❌ Scenario 3: you said {'paradigm': '?', 'task': '?'}, answer key says {'paradigm': 'reinforcement', 'task': 'n/a'}
❌ Scenario 4: you said {'paradigm': '?', 'task': '?'}, answer key says {'paradigm': 'supervised', 'task': 'classification'}


### 🔍 Notice

Could our dice-roll function from Step 1 be turned into a tiny "model"? What would it mean to *predict* the next roll — and how would we know if that prediction was any good?

<details>
<summary>💡 <b>What's going on here?</b></summary>

Yes — that's exactly what the next section builds. "Predicting" here just means using whatever you've already seen to make a guess about something you haven't seen yet. "Any good" means comparing that guess against what actually happened, afterwards. Both ideas get made fully concrete in code in the next section.

</details>

## Step 3 — Reconstruct: A Naive "Model"

Let's build the simplest possible predictor, and a way to score it. This is a **baseline predictor**: a simple rule whose performance provides a starting point for comparison. It learns which value was most frequent in the supplied past data.

We'll start with the scoring function, because it's the easier of the two to read.

For our independent fair-die simulation, no past-roll pattern makes a face more likely next time: the expected accuracy is 1/6, about 16.7%. We use it to practise prediction and scoring, not to demonstrate a learnable relationship.

### ▶️ Run & Read: Scoring Predictions

Here is a working function that scores a set of predictions: it counts how many of them exactly matched what really happened. Run it, then read it line by line.

> 🐍 **Python note:** `zip(true_rolls, predicted_rolls)` walks through two lists **together, position by position** — first `(true_rolls[0], predicted_rolls[0])`, then `(true_rolls[1], predicted_rolls[1])`, and so on. Writing `for t, p in zip(...)` gives the two values in each pair the names `t` (true) and `p` (predicted) inside the loop.

In [ ]:
def score_predictions(true_rolls, predicted_rolls):
    """Count how many predictions exactly matched the true value."""
    if len(true_rolls) != len(predicted_rolls):
        raise ValueError("Provide one prediction for each true value; the lists must be equally long.")
    matches = 0                                      # start counting from zero
    for t, p in zip(true_rolls, predicted_rolls):    # take one true roll and its prediction at a time
        if t == p:                                   # did the prediction match what really happened?
            matches = matches + 1                    # yes: add one to the count
    return matches                                   # after the loop, hand back the total

# Try it on two short lists you can check by eye
print(score_predictions([4, 2, 6, 1], [4, 5, 6, 3]))

2


**Tracing it by hand.** Here is what happens each time round the loop for that test:

| Time round the loop | `t` | `p` | `t == p`? | `matches` afterwards |
|---|---|---|---|---|
| 1st | 4 | 4 | True | 1 |
| 2nd | 2 | 5 | False | 1 |
| 3rd | 6 | 6 | True | 2 |
| 4th | 1 | 3 | False | 2 |

Once the loop has finished, the function returns `matches`, which is `2` — the number printed above. Tracing a loop like this, one row per time round, is one of the most reliable ways to understand code you didn't write.

### 🔧 Tweak It: Change What Counts as Correct

The cell below starts as an exact copy of `score_predictions()`, renamed `score_tweak()` so the original stays safe. Make each change **one at a time**, and before each run, predict what will be printed.

1. Change `==` to `!=` (which means "is not equal to"). *Predict:* what will the function be counting now? What will the small test print?
2. Change it back to `==`. Then change the whole line `if t == p:` to `if abs(t - p) <= 1:`. Now a prediction counts as correct if it's **within 1** of the real roll. *Predict:* will the score go up, go down, or could it go either way?
3. 📝 **Note this down:** with change 2 in place, what **score** does the last line print? You'll be asked for it on the course page at the end of this module.

> 🐍 **Python note:** `abs(x)` gives the size of a number, ignoring its sign: `abs(-2)` and `abs(2)` are both `2`. So `abs(t - p) <= 1` means "`t` and `p` are at most 1 apart".

In [ ]:
def score_tweak(true_rolls, predicted_rolls):
    matches = 0
    for t, p in zip(true_rolls, predicted_rolls):
        if t == p:                                   # <- the line to tweak
            matches = matches + 1
    return matches

print("Small test:", score_tweak([4, 2, 6, 1], [4, 5, 6, 3]))

true_rolls = roll_dice(10, seed=99)
guesses = roll_dice(10, seed=5)
print("True rolls:", true_rolls)
print("Guesses:   ", guesses)
print("Score:     ", score_tweak(true_rolls, guesses))

Small test: 2
True rolls: [4, 4, 2, 5, 2, 2, 2, 2, 1, 3]
Guesses:    [5, 3, 6, 3, 6, 6, 6, 5, 1, 4]
Score:      1


<details>
<summary>💡 <b>What's going on here?</b></summary>

1. **The misses.** With `!=`, the function counts the predictions that *didn't* match. For the small test that's 2: there are four predictions and two matches, so two misses. Matches plus misses always add up to the number of predictions.
2. **Up, or stay the same — never down.** Every exact match is still within 1, so nothing that counted before stops counting; some near-misses start counting too.

That second change is a genuine machine learning decision in disguise. The same predictions got two different scores, just from changing what "correct" means. Choosing *how* to measure a model is part of the job, and you'll see much more of it in Module 4.

</details>

### 🧩 Put It in Order: Counting Each Roll

A predictor that guesses "the most common roll so far" first needs to **count** how often each value appears. Below are the lines of a function called `count_each()`, which does exactly that — but they've been shuffled, and **one extra line that doesn't belong** has been slipped in.

`count_each([4, 1, 4, 6, 4, 1])` should give back `{4: 3, 1: 2, 6: 1}`: the value 4 appeared three times, 1 twice, and 6 once.

```
A │             counts[r] = counts[r] + 1
B │ def count_each(rolls):
C │     return counts
D │         else:
E │     counts = {}
F │         counts = {}
G │     for r in rolls:
H │             counts[r] = 1
I │         if r in counts:
```

The indentation (how far each line is pushed in from the left) is already correct. You only need to choose the order, and spot the one line to leave out.

**Your task:** in the cell below, list the letters in order. Then run it: it puts your lines together, shows you the result, and tests it.

> 🐍 **Python note:** `counts = {}` makes an empty dictionary. `if r in counts:` asks "is `r` already one of the dictionary's keys?". If it is, `counts[r] = counts[r] + 1` adds one to its count; if not, `counts[r] = 1` adds it as a new key with a count of 1. `else:` means "otherwise" — the lines under it run only when the `if` test was false.

In [ ]:
# 🧩 List the letters in order, as strings, leaving out the one line that doesn't belong.
# For example:  my_order = ["C", "A", "B"]
my_order = []

check_puzzle("counts", my_order)

✏️  my_order is still empty. Type your letters between the square brackets, e.g. ["C", "A", "B"], then run this cell again.


<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

Every function starts with its `def` line. Then ask: what has to exist **before** the loop can use it? What happens **inside** the loop, once for each roll? And what happens just once, **after** the loop has finished? The indentation is a clue too: indentation groups statements into a block introduced by a line ending in a colon `:`; the whole block belongs to that function, loop or condition.

</details>

### ▶️ Run & Read: A Shortcut for Counting

Counting how often each value appears is so common that Python has a ready-made tool for it: `Counter`, which you imported in Step 0. It does the same job as the `count_each()` function you just put together. The cell below uses it **one step at a time**, printing the result of each step, so you can see exactly how to get from a list of rolls to the single most common value.

> 🐍 **Python note:** `(4, 3)` is a **tuple**: an ordered collection whose entries cannot be reassigned after creation. The comma forms the tuple; the brackets make its grouping clear. You take items out of it with `[0]`, `[1]` and so on, exactly as with a list.

In [ ]:
rolls = [4, 1, 4, 6, 4, 1]

counts = Counter(rolls)          # count each value, just like count_each()
print("Step 1 — Counter(rolls):  ", counts)

top = counts.most_common(1)      # the single most common value, as a list of one (value, count) pair
print("Step 2 — .most_common(1): ", top)

pair = top[0]                    # [0] takes the first (and only) pair out of the list
print("Step 3 — top[0]:          ", pair)

value = pair[0]                  # [0] again takes the first item of the pair: the value itself
print("Step 4 — pair[0]:         ", value)

print()
print("All four steps on one line:", Counter(rolls).most_common(1)[0][0])

Step 1 — Counter(rolls):   Counter({4: 3, 1: 2, 6: 1})
Step 2 — .most_common(1):  [(4, 3)]
Step 3 — top[0]:           (4, 3)
Step 4 — pair[0]:          4

All four steps on one line: 4


### 🎯 Try It Yourself: The Naive Predictor

Now write `predict_next_roll()`. Given a list of past rolls, it should return the single value that appeared **most often**. That one value becomes our "model's" guess for every future roll.

You can use `Counter`, as in the cell above, or the `count_each()` function you put together — either way works. The test at the bottom checks your function on a small list. The cell after it uses your function as a model: 50 "training" rolls decide the guess, and 10 brand-new rolls test it.

> 🐍 **Python note:** `[guess] * len(test_rolls)` makes a list containing `guess` repeated once for every test roll — for example, `[4] * 3` is `[4, 4, 4]`.

**Input and output:** a non-empty list of integer rolls → one integer roll value. If values tie for highest frequency, either tied value is acceptable; Counter chooses the one encountered first. Use `return`, not just `print`.

In [ ]:
def predict_next_roll(previous_rolls):
    """A very naive 'model': always predict the most common past roll."""
    # 🔧 TODO: return the single most frequently occurring value in previous_rolls
    pass  # <- replace this line


# Test it — do not edit below this line
print("Guess for [4, 1, 4, 6, 4, 1]:", predict_next_roll([4, 1, 4, 6, 4, 1]), "  (expected 4)")

Guess for [4, 1, 4, 6, 4, 1]: None   (expected 4)


<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

Look back at the ▶️ **Run & Read: A Shortcut for Counting** cell just above. Its Steps 1 to 4 already go from a list of rolls to the single most common value. Your function needs to do the same thing to `previous_rolls`, and `return` the value at the end.

</details>

In [ ]:
# Use your predict_next_roll() as a model: 50 training rolls decide the guess, and 10 new rolls test it
train_rolls = roll_dice(50, seed=7)
guess = predict_next_roll(train_rolls)
test_rolls = roll_dice(10, seed=99)
predicted_rolls = [guess] * len(test_rolls)
score = score_predictions(test_rolls, predicted_rolls)

print(f"Our naive model always guesses: {guess}")
print(f"True rolls (new data):          {test_rolls}")
print(f"It got {score}/{len(test_rolls)} correct on new data.")

Our naive model always guesses: None
True rolls (new data):          [4, 4, 2, 5, 2, 2, 2, 2, 1, 3]
It got 0/10 correct on new data.


That `score` is an evaluation metric: here it counts correct guesses. Dividing by the number of guesses gives accuracy, the fraction correct. A **loss** measures error and supplies an objective to minimise during training. The distinction is their purpose, not one prediction versus many: losses can be averaged, and one quantity can serve as both loss and metric. Our baseline chooses the most frequent training value directly; it does not run an iterative optimiser. See Lesson 1.2's worked example for parameter adjustment.

### ▶️ Run & Read: One Model, Ten Batches of New Rolls

Your naive model was scored on one batch of ten new rolls, made with `seed=99`. The cell below scores the **same** model — the same `guess` — on ten different batches. Before you run it, predict: will all ten scores be the same?

In [ ]:
# Score the same model on ten different batches of new rolls
if guess is None:                                                         # has The Naive Predictor been finished?
    print("Finish 🎯 The Naive Predictor above, run it, then run this cell again.")  # if not, say what to do
else:                                                                     # otherwise, score ten batches
    for test_seed in range(99, 109):                                      # seeds 99 to 108: ten different batches
        new_rolls = roll_dice(10, seed=test_seed)                         # ten new rolls for this batch
        new_score = score_predictions(new_rolls, [guess] * len(new_rolls))  # same model: same guess every time
        print(f"Batch made with seed={test_seed}: {new_score}/{len(new_rolls)} correct")  # show this batch's score

Finish 🎯 The Naive Predictor above, run it, then run this cell again.


### 🔍 Notice

Nothing about the model changed between those ten lines. So which score is the model's *real* one? And if you changed the model and its score went up by 1, could you tell whether your change had helped?

<details>
<summary>💡 <b>What's going on here?</b></summary>

No single batch gives the model's "real" score. Each batch is a different random sample of rolls, so the score moves around even though the model stays exactly the same. A score that goes up by 1 could easily be luck.

Real machine learning has the same problem. Which examples end up in the test data is often decided at random, and training itself can use randomness. Two habits help. To compare two versions of a model fairly, test them on exactly the same data: that's one job the seeds in this notebook do. To see how much a result depends on luck, repeat it with several different seeds and look at the spread, as you just did. The Reproducibility & Random Seeds lesson comes back to both.

</details>

### 🌟 Stretch Challenge (optional): Compare Two Baseline Predictors

`predict_next_roll()` always guesses the single most common value. Write a variant, `predict_next_roll_v2()`, that instead guesses the **second-most-common** value — then compare its score against the original on the same test data. Does it do better or worse? Is that what you expected?

This is optional — skip ahead to Step 4 any time.

**Interpret carefully:** for an independent fair die, every face has probability 1/6 on the next roll. Past rolls do not make the next roll predictable. A higher score on just ten test rolls may be chance; it does not establish a better strategy.

In [ ]:
def predict_next_roll_v2(previous_rolls):
    """Variant: predict the SECOND most common past roll."""
    # 🔧 TODO: Counter.most_common(2) returns the top two (value, count) pairs
    pass  # <- replace this line


guess_v2 = predict_next_roll_v2(train_rolls)
predicted_rolls_v2 = [guess_v2] * len(test_rolls)
score_v2 = score_predictions(test_rolls, predicted_rolls_v2)

print(f"Original model guessed {guess}, scoring {score}/{len(test_rolls)}")
print(f"v2 model guessed {guess_v2}, scoring {score_v2}/{len(test_rolls)}")

Original model guessed None, scoring 0/10
v2 model guessed None, scoring 0/10


<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

This is the same idea as `predict_next_roll()` just above, but you want the **second**-most-common value instead of the most common one. `most_common()` can give back more than just the single top result if you ask it to: think about how many results you'd need, and which position in them holds the value you want.

</details>

## Step 4 — 📚 Read on the Course Page: Vectors & Comparing Data

Suppose two students each roll a die 8 times and want to compare their sequences. How would you measure how "similar" or "different" their results are?

Head to the [**Vectors & Comparing Data** lesson](https://fair.aisingapore.org/lesson/vectors-comparing-data-3/) now — it builds up L1 distance, L2 distance, and cosine similarity step by step from a worked numeric example, introducing the Σ (sigma) notation only once you already understand what it represents.

Come back here once you've read it — you're about to see exactly what you worked through by hand turned into code, change it, and then build parts of it yourself.

### ▶️ Run & Read: L1 and L2 Distance

Here are working L1 and L2 distance functions, one cell each, written to follow the exact steps from the course page. The first test uses the course page's own example, A = [3, 1, 4] and B = [5, 1, 2], so you can check the answers against it: 4, and about 2.83.

> 🐍 **Python note:** `x ** 2` means "x squared". `math.sqrt(x)` takes a square root. `zip(v1, v2)` pairs up matching positions, exactly as in `score_predictions()`. Both functions add up a running `total` inside a loop — this "start at zero, add something each time round, return the total" pattern is precisely what the Σ symbol on the course page means.

**Input contract:** use two non-empty numeric lists with the same number of entries. Each position must describe the same quantity in both lists. The supplied input check reports a helpful error otherwise; `zip()` alone would silently ignore extra entries.

In [ ]:
def l1_distance(v1, v2):
    """Manhattan (L1) distance: add up the absolute difference at each position."""
    _require_pair(v1, v2)                  # both lists must be non-empty and equally long
    total = 0                              # running total, starts at zero
    for a, b in zip(v1, v2):               # one position at a time: a from v1, b from v2
        total = total + abs(a - b)         # add how far apart they are, ignoring the sign
    return total                           # the sum of all the position-by-position distances


A = [3, 1, 4]
B = [5, 1, 2]
print("The course page's example")
print("  L1 distance:", l1_distance(A, B))

seq_a = roll_dice(8, seed=1)
seq_b = roll_dice(8, seed=2)
print("\nTwo sequences of dice rolls")
print("  seq_a:", seq_a)
print("  seq_b:", seq_b)
print("  L1 distance:", l1_distance(seq_a, seq_b))

The course page's example
  L1 distance: 4

Two sequences of dice rolls
  seq_a: [2, 5, 1, 3, 1, 4, 4, 4]
  seq_b: [1, 1, 1, 3, 2, 6, 6, 3]
  L1 distance: 11


**Tracing `l1_distance(A, B)` by hand** — compare this with Steps 1 and 2 on the course page. Same numbers, same order:

| Position | `a` | `b` | `abs(a - b)` | `total` afterwards |
|---|---|---|---|---|
| 1 | 3 | 5 | 2 | 2 |
| 2 | 1 | 1 | 0 | 2 |
| 3 | 4 | 2 | 2 | 4 |

In [ ]:
def l2_distance(v1, v2):
    """Euclidean (L2) distance: square each difference, add them up, square-root once at the end."""
    _require_pair(v1, v2)                  # both lists must be non-empty and equally long
    total = 0                              # running total of squared differences
    for a, b in zip(v1, v2):               # one position at a time
        total = total + (a - b) ** 2       # add the squared difference
    return math.sqrt(total)                # ONE square root, after the loop has finished


print("The course page's example")
print("  L2 distance:", round(l2_distance(A, B), 3))
print("\nTwo sequences of dice rolls")
print("  L2 distance:", round(l2_distance(seq_a, seq_b), 3))

The course page's example
  L2 distance: 2.828

Two sequences of dice rolls
  L2 distance: 5.196


### 🔧 Tweak It: Averages, and a Common Mistake

Two copies to experiment with, one cell each: `mean_difference()` starts as a copy of `l1_distance()`, and `l2_tweak()` starts as a copy of `l2_distance()`. Make each change, and predict before you run.

1. In `mean_difference()`, change `return total` to `return total / len(v1)`. This divides by the number of positions, giving the **average** difference per position. *Predict:* what will it print for A and B?
2. In `l2_tweak()`, make the mistake the course page warned about. Change `total = total + (a - b) ** 2` to `total = total + math.sqrt((a - b) ** 2)`, and change `return math.sqrt(total)` to `return total`. Now the square root happens once per position, instead of once at the end. *Predict:* will the answer for A and B still be about 2.83?
3. 📝 **Note this down:** with change 1 in place, what does the first cell's last line print for `mean_difference(seq_a, seq_b)`? You'll be asked for it on the course page at the end of this module.

In [ ]:
def mean_difference(v1, v2):
    total = 0
    for a, b in zip(v1, v2):
        total = total + abs(a - b)
    return total                             # <- change 1


print("mean_difference(A, B):", round(mean_difference(A, B), 3))
print("mean_difference(seq_a, seq_b):", round(mean_difference(seq_a, seq_b), 3))

mean_difference(A, B): 4
mean_difference(seq_a, seq_b): 11


In [ ]:
def l2_tweak(v1, v2):
    total = 0
    for a, b in zip(v1, v2):
        total = total + (a - b) ** 2         # <- change 2 (this line...)
    return math.sqrt(total)                  # <- change 2 (...and this one)


print("l2_tweak(A, B):", round(l2_tweak(A, B), 3), "   (the correct L2 distance is", round(l2_distance(A, B), 3), ")")

l2_tweak(A, B): 2.828    (the correct L2 distance is 2.828 )


<details>
<summary>💡 <b>What's going on here?</b></summary>

1. **About 1.333** (4 ÷ 3). Each pair must have the same number of matching positions. Dividing by that number gives the mean absolute difference. This helps compare average errors over different numbers of comparable observations; it does not make unrelated features or units comparable. You'll meet exactly this idea again in Module 4, where it's called the *mean absolute error*.
2. **No — it prints 4.0, the L1 distance.** Taking the square root of a squared difference gives its absolute value: √((−2)²) is 2, the plain distance. So taking the root too early quietly turns L2 back into L1. That's why the square root has to be taken once, at the very end. Seeing a wrong-but-reasonable-looking number like this is a good reminder to check your code against a small example you've worked out by hand.

</details>

### 🧩 Put It in Order: The Dot Product

Cosine similarity needs two ingredients: the **dot product** of the two vectors, and each vector's **magnitude**. Here are the lines of `dot_product()`, shuffled, plus one line that doesn't belong. It should multiply each pair of matching positions and add up the results: for A = [3, 1, 4] and B = [5, 1, 2], that's 15 + 1 + 8 = 24, as on the course page.

```
A │     for a, b in zip(v1, v2):
B │         total = a * b
C │     return total
D │ def dot_product(v1, v2):
E │         total = total + a * b
F │     total = 0
```

**Your task:** list the letters in order in the cell below, leaving out the line that doesn't belong, then run it.

In [ ]:
# 🧩 List the letters in order, as strings, leaving out the one line that doesn't belong.
# For example:  my_order = ["C", "A", "B"]
my_order = []

check_puzzle("dot", my_order)

✏️  my_order is still empty. Type your letters between the square brackets, e.g. ["C", "A", "B"], then run this cell again.


<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

Look at `l1_distance()` in the ▶️ Run & Read cell above: start a running total **before** the loop, add something to it **inside** the loop, and return it **after** the loop. This function has exactly the same shape; only what gets added is different. And watch out for the difference between *adding to* a total and *replacing* it.

</details>

### 🎯 Try It Yourself: A Vector's Magnitude

Now write `magnitude()`, the second ingredient. It takes **one** vector and returns its length: square each value, add up the squares, then take the square root **once**, at the end. For A = [3, 1, 4], that's √(9 + 1 + 16) = √26 ≈ 5.10, exactly as on the course page.

**Input and output:** a non-empty numeric list → a non-negative number. Magnitude means distance from the origin (the zero vector), not the number of entries. For `[3, 4]`, the magnitude is 5; distance from `[3, 4]` to itself is 0.

In [ ]:
def magnitude(v):
    """The length of vector v: square each value, add them up, square-root once at the end."""
    # 🔧 TODO: same shape as l2_distance(), but for a single vector
    pass  # <- replace this line


# Test your function — do not edit below this line
print("magnitude([3, 4]):   ", magnitude([3, 4]), "  (expected 5.0)")
assert magnitude([3, 4]) == 5.0 and round(magnitude([3, 1, 4]), 3) == 5.099, \
    "Not quite: square each value, add them all up, then take ONE square root at the end (and return it)."
print("magnitude([3, 1, 4]):", round(magnitude([3, 1, 4]), 3), "  (expected 5.099)")
print("✅ Your magnitude() function works.")

magnitude([3, 4]):    None   (expected 5.0)


AssertionError: Not quite: square each value, add them all up, then take ONE square root at the end (and return it).

<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

`l2_distance()` above already does almost exactly this, but for the *difference* between two vectors. Here there's only one vector, so the loop can go through its values directly, and each value itself is what gets squared, not a difference. Keep the square root outside the loop.

</details>

### ▶️ Run & Read: Cosine Similarity

With both ingredients ready, cosine similarity is just Step 3 from the course page: divide the dot product by the **product** of the two magnitudes. For A and B you should get about 0.86.

*If this cell says `dot_product` or `magnitude` is not defined, go back and finish the 🧩 puzzle and the 🎯 exercise above first — this cell is built from both of them.*

**Input contract:** matching, non-empty lists representing non-zero vectors. For `[1, 2]` and `[2, 4]`, cosine similarity is 1 because direction matches, even though their values and magnitudes differ. A zero vector has no direction.

In [ ]:
def cosine_similarity(v1, v2):
    """How similar in direction two vectors are, from -1 to 1."""
    _require_pair(v1, v2)          # compare matching positions
    dot = dot_product(v1, v2)        # Step 1: the dot product (your 🧩 puzzle)
    mag1 = magnitude(v1)             # Step 2: each vector's own length (your 🎯 function)
    mag2 = magnitude(v2)
    if mag1 == 0 or mag2 == 0:
        raise ValueError("Cosine similarity needs two non-zero vectors: a zero vector has no direction.")
    return dot / (mag1 * mag2)       # Step 3: divide by the PRODUCT of the two magnitudes


print("The course page's example:", round(cosine_similarity(A, B), 3))
print("seq_a vs seq_b:           ", round(cosine_similarity(seq_a, seq_b), 3))

### 🔍 Notice

Try comparing `seq_a` against itself (`l2_distance(seq_a, seq_a)` and `cosine_similarity(seq_a, seq_a)`). What do you get, and does it make sense given the definitions above?

> **Looking ahead:** In Module 2, you'll redo these exact same calculations using NumPy arrays instead of plain Python lists and loops — same maths, dramatically less code, and it scales to far larger sequences.

<details>
<summary>💡 <b>What's going on here?</b></summary>

L2 distance between a vector and itself is 0 — there's no difference at any position, so every squared difference is 0, and the square root of 0 is still 0.

Cosine similarity between a non-zero vector and itself is 1. A zero vector has magnitude zero and no direction, so the formula is undefined for it: division by zero is not allowed.

These are useful checks for the measures taught here: L1 and L2 give zero for identical vectors; cosine similarity gives 1 for identical non-zero vectors.

</details>

### 🌟 Stretch Challenge (optional): Rank by Similarity

Given one reference sequence and three candidate sequences, use `cosine_similarity()` to work out which candidate is most similar to the reference, and print all three ranked from most to least similar.

> 🐍 **Python note:** `{name: cosine_similarity(reference, seq) for name, seq in candidates.items()}` is a **dictionary comprehension** — the dictionary equivalent of the list comprehension from Step 1. `candidates.items()` gives you each `(name, seq)` pair in the `candidates` dictionary, one at a time, and this builds a new dictionary mapping each `name` to its similarity score. `sorted(..., key=lambda pair: pair[1], reverse=True)` sorts those `(name, score)` pairs by score, highest first — `lambda pair: pair[1]` is a tiny, unnamed function that just says "look at the second item of each pair" (the score) when deciding the order.

This is optional — skip ahead to Step 5 any time.

In [ ]:
reference = roll_dice(8, seed=10)
candidates = {
    "A": roll_dice(8, seed=11),
    "B": roll_dice(8, seed=12),
    "C": roll_dice(8, seed=13),
}

# 🔧 TODO: compute cosine similarity between the reference and each candidate
similarities = {}  # <- fill this dictionary: name -> similarity score

# Sort candidates from most similar (highest score) to least similar
ranked = sorted(similarities.items(), key=lambda pair: pair[1], reverse=True)

print(f"Reference: {reference}\n")
for name, score in ranked:
    print(f"{name}: {candidates[name]}  —  similarity {round(score, 3)}")

<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

You already have a working `cosine_similarity()` function from earlier in this notebook — you don't need to write any new similarity maths here. You just need to call it once for each of the three candidates and store each result in the `similarities` dictionary, keyed by the candidate's name. A plain `for` loop over the items of the `candidates` dictionary works fine, giving you each name and its sequence in turn.

</details>

## Step 5 — ✍️ Explain

In **one short paragraph**, write your own answer to this question (edit the text cell below):

> *What role does a loss play during training, and what role does an evaluation metric play when judging predictions? Can the same calculation serve both roles? Use Lesson 1.2's example to explain.*

This is not graded — it's here to make you put the idea in your own words before moving on. There's no need to "get it perfect"; a genuine attempt is what matters.

After writing, compare with the **Loss and metrics** explanation in Solutions & Explanations. Check that you name the training role, the evaluation role and the possibility of using the same calculation for both.

**Your answer:**

*(double-click this cell to edit, then run it to render your text)*

...

## Step 6 — 📚 Read on the Course Page: Reproducibility & Random Seeds

Throughout this notebook, `roll_dice()` and its copies have often been called with an extra `seed=...` argument — in Step 1's Run & Read and Tweak It cells, in the naive-model section, and again when comparing vectors — and every time, your results have matched exactly what's shown here, even across completely separate runs. You've been relying on that without a name for it.

In a moment, this notebook is going to check your work automatically and hand you a Module 1 completion code. Before you get there, it's worth understanding what's actually going on.

Head to the [**Reproducibility & Random Seeds** lesson](https://fair.aisingapore.org/lesson/reproducibility-random-seeds-3/) now — it explains pseudo-randomness, what a seed actually does, and the other random-number tools you've been using throughout this notebook without a name for them yet.

Come back here once you've read it — the next few things you do (a quick reproducibility check, a short puzzle, then a short exercise) lead straight into the Check cell that verifies your work.

In [ ]:
# 🔧 TODO (already filled in): call roll_dice with the SAME seed twice and confirm the results match.
rolls_c = roll_dice(10, seed=42)
rolls_d = roll_dice(10, seed=42)
print("Run with seed=42 (first): ", rolls_c)
print("Run with seed=42 (second):", rolls_d)
print("Identical?", rolls_c == rolls_d)

assert rolls_c == rolls_d, "Rolls should match when using the same seed!"
print("Same seed, same rolls: the two runs match.")

### Reconstruct: The Other Random-Number Tools

The course page's table listed more tools than `random.randint()`. The cell below uses three of them, under the same seed twice, so you can see that every one of them is reproducible:

- `random.random()` — a floating-point number at least 0.0 and less than 1.0
- `random.uniform(a, b)` — a floating-point sample from a uniform distribution between the bounds; equal-width intervals have equal probability in the ideal distribution
- `random.gauss(mu, sigma)` — a floating-point sample from a normal distribution; `mu` is its mean (centre) and `sigma` its standard deviation (a measure of spread). Here sigma is positive.

In [ ]:
for attempt in (1, 2):
    random.seed(5)
    print(f"Attempt {attempt}: random() = {random.random():.4f}   "
          f"uniform(1, 10) = {random.uniform(1, 10):.4f}   "
          f"gauss(0, 1) = {random.gauss(0, 1):.4f}")

### 🧩 Put It in Order: A Die With Any Number of Sides

Now that you know what a seed does, here is one last puzzle before you write a generator of your own. These are the lines of `roll_any_die()`, which rolls a die with any number of `sides` — reproducibly, whenever a seed is given. As before, they've been shuffled, and one line doesn't belong.

```
A │     for _ in range(n_rolls):
B │         random.seed(seed)
C │     return rolls
D │         rolls = random.randint(1, sides)
E │ def roll_any_die(n_rolls, sides, seed=None):
F │     rolls = []
G │         rolls.append(random.randint(1, sides))
H │     if seed is not None:
```

**Your task:** list the letters in order in the cell below, leaving out the line that doesn't belong, then run it. The checker runs your function twice with the same seed, so it will notice if the two runs don't match.

In [ ]:
# 🧩 List the letters in order, as strings, leaving out the one line that doesn't belong.
# For example:  my_order = ["C", "A", "B"]
my_order = []

check_puzzle("dice", my_order)

<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

Start with the `def` line. `roll_dice_long()` in Step 1 has almost exactly this shape; the only differences are the function's name, the extra `sides` parameter, and what goes inside `random.randint()`. Remember what the course page said about **when** a seed has to be set.

</details>

### 🎯 Try It Yourself: Build Your Own Reproducible Generator

`roll_dice()` and `roll_any_die()` use `random.randint()` to generate whole numbers. Now write your own function using `random.uniform()` instead, to generate **measurements** — random decimal numbers between two bounds you choose.

**Your task:** complete `generate_measurements()` below so that it follows the exact same reproducibility pattern as `roll_any_die()` — set the seed if one is given, then generate `n` values with `random.uniform(low, high)`. You can build the list with a `for` loop, as in the puzzle, or with a one-line list comprehension, as in `roll_dice()` — either is fine.

Run the cell once you've written it. If your function is correct, the two runs will match and you'll see a ✅.

**Input and output:** a non-negative whole-number count `n`, bounds with `low <= high`, and an optional seed → a list of `n` floating-point samples. With `n=0`, return `[]`. Set a supplied seed once before drawing; with `seed=None`, continue the current generator state. A float is Python's finite-precision representation of a real number.

In [ ]:
def generate_measurements(n, low, high, seed=None):
    """Generate n random floats between low and high, reproducible via seed."""
    # 🔧 TODO: follow the same pattern as roll_any_die(), but use random.uniform(low, high)
    pass  # <- replace this line


# Test your function — do not edit below this line
try:
    _check_measurements(generate_measurements)
    print("✅ Your generator matches the reference for several seeds, bounds and lengths.")
except Exception as err:
    print(f"❌ {type(err).__name__}: {err}")
    print("Check the hint, revise the function above and rerun this cell.")


<details>
<summary>🔒 <b>Stuck? Hint</b></summary>

This is almost identical to `roll_any_die()` from the puzzle just above — same "set the seed first, then build a list with a loop" pattern. The differences: each value comes from `random.uniform()` instead of `random.randint()`, between `low` and `high`, and the loop repeats `n` times instead of `n_rolls` times.

</details>

## Step 7 — ✅ Check: Automated Verification

Run the cell below. It re-checks the key pieces of code you worked with in this notebook — including the functions you put in order and the ones you wrote yourself. If everything passes, you’ll get a Module 1 completion code. Paste that into the Module 1 completion question on the course page to proceed.

In [ ]:
# --- Automated checks: do not modify this cell ---
checks_passed = True


def run_check(label, test):
    """Report every result, including errors, so you know what to revisit."""
    global checks_passed
    state = random.getstate()
    try:
        assert test(), "The result did not match the expected value. Revisit this activity and its examples."
        print(f"✅ {label} check passed.")
    except Exception as err:
        checks_passed = False
        print(f"❌ {label}: {type(err).__name__}: {err}")
    finally:
        random.setstate(state)


def check_dice():
    for n, sides, seed in [(5, 20, 3), (7, 6, 0), (0, 8, 9)]:
        rng = random.Random(seed)
        expected = [rng.randint(1, sides) for _ in range(n)]
        assert roll_any_die(n, sides, seed=seed) == expected, "Revisit the dice puzzle: seed once, then append each roll."
    return roll_dice(10, seed=42) == roll_dice(10, seed=42)


def check_predictor():
    for values in [[3, 3, 3, 5, 1], [6, 1, 6, 2, 6], [2], [1, 2, 1, 2]]:
        counts = Counter(values)
        answer = predict_next_roll(values)
        assert isinstance(answer, int) and answer in counts, "Return a roll value, not its count or a tuple."
        assert counts[answer] == max(counts.values()), "Return one of the most frequent values."
    return True


vector_cases = [([1, 2, 3], [4, 6, 8], 40), ([3, 1, 4], [5, 1, 2], 24), ([-2, 0], [1, 4], -2), ([2], [2], 4)]
run_check("Dice and reproducibility", check_dice)
run_check("count_each", lambda: all(count_each(x) == dict(Counter(x))
          for x in [[4, 1, 4, 6, 4, 1], [2, 2, 2], [6], []]))
run_check("dot_product", lambda: all(dot_product(a, b) == dot for a, b, dot in vector_cases))
run_check("magnitude", lambda: all(math.isclose(magnitude(v), expected, abs_tol=1e-12)
          for v, expected in [([3, 4], 5), ([-3, 0, 4], 5), ([0, 0], 0), ([2], 2), ([1, 2, 2], 3)]))
run_check("Cosine similarity", lambda: all(math.isclose(cosine_similarity(a, b), expected, abs_tol=1e-12)
          for a, b, expected in [([1, 2], [2, 4], 1), ([1, 0], [0, 1], 0),
                                  ([1, 2], [-1, -2], -1), ([3, 1, 4], [5, 1, 2], 24/math.sqrt(780))]))
run_check("Vector input guidance", lambda: all(_expect_value_error(f, [1, 2], [1])
          for f in [l1_distance, l2_distance, cosine_similarity])
          and _expect_value_error(cosine_similarity, [0, 0], [1, 2]))
run_check("predict_next_roll", check_predictor)
run_check("generate_measurements", lambda: _check_measurements(generate_measurements))

# Its original constants stay unchanged so existing quiz answers remain valid.
expected_l1 = 12
expected_l2 = round(math.sqrt(3**2 + 4**2 + 5**2), 3)
print()
if checks_passed:
    token_source = f"FAIR-M1-{expected_l1}-{expected_l2}"
    token = hashlib.sha256(token_source.encode()).hexdigest()[:8].upper()
    print("🎉 All checks passed for the tested cases!")
    print(f"Your Module 1 completion code: {token}")
    print("Enter this exact code into the Module 1 completion question on the course page.")
else:
    print("⚠️ Revisit the named activity, fix and rerun its cell, then run this Check cell again.")


## Well done!

You've just:
- Met the core vocabulary of the field: AI, ML, DL, GenAI, AGI, and the four learning paradigms
- Built and scored your first (very naive!) predictive model
- Represented data as vectors and compared them three different ways
- Learned why reproducibility matters, and how to control it with seeds
- Read, tweaked, reassembled, and written your own Python functions — the same four moves you'll keep using for the rest of FAIR

**Before you go:** head back to the course page and complete the short Module 1 completion questions there. You'll need your completion code and the three 📝 numbers you noted down.

**Next up: Module 2 — Arrays in Action**, where you'll redo some of today's calculations with NumPy and meet probability distributions and your first data visualisations.

## 🔭 Sneak Peek: What's Coming

The required activities have used Python and its standard library. The next cell introduces third-party libraries. Before you go, here's a 30-second look at where all of this is heading.

The cell below uses a library called **scikit-learn** (plus a chart from **Matplotlib**) — neither of which you've learned yet. That's completely fine and totally intentional. Don't try to understand every line. Just run it and watch what happens.

*(This part isn't graded, isn't checked, and isn't building toward anything you need to remember right now — it's just a preview.)*

In [ ]:
# This cell uses tools you haven't learned yet (scikit-learn, matplotlib) — just run it and watch.
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

digits = load_digits()  # 1,797 real images of handwritten digits, 0-9

# Split images and labels together, so we can show a real picture afterwards
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    digits.data, digits.target, digits.images, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"This model correctly classifies {accuracy:.0%} of the held-out test images in this split.")

# Show one example the model has never seen, and its guess
plt.imshow(img_test[0], cmap="gray")
plt.title(f"Model's guess: {model.predict([X_test[0]])[0]}   (actually: {y_test[0]})")
plt.axis("off")
plt.show()

That's a real, working handwriting-reading model, built in about ten lines of code. Right now it's magic — a black box that just *works*.

By the end of Module 6, that magic will be gone: you'll have trained a neural network on this exact digits dataset yourself, from scratch, and you'll understand every step of how it learns. (If you later continue into FAIR's Computer Vision track, you'll go one step further and build a network designed specifically for reading images.) Keep an eye out for this dataset — it's coming back.

## 🔑 Solutions & Explanations

Here's the full worked solution to every exercise and puzzle in this notebook, in the order they appear, each with a plain-language explanation of how it works and why. If an exercise gave you trouble, or you'd just like to double-check your reasoning, this is the place — there's no penalty for looking, and reading a solution carefully is a completely legitimate way to learn.

**These solutions are displayed as text, not executable cells.** Reading them cannot replace your functions in memory. If you use a solution, revise your original exercise cell deliberately and rerun it before using Check. The optional ranking solution can be read without having run the optional challenge.

### Classify These Scenarios

```python
my_answers = {
    1: {"paradigm": "supervised", "task": "regression"},   # known prices to learn from -> supervised; predicting a number -> regression
    2: {"paradigm": "unsupervised", "task": "n/a"},         # no topic labels given at all
    3: {"paradigm": "reinforcement", "task": "n/a"},        # learns from trial, error, and reward
    4: {"paradigm": "supervised", "task": "classification"}, # known spam/not-spam labels -> supervised; predicting a category -> classification
}
```

**How to work through each one:** ask two questions in order. First, **are there labelled examples with a known correct answer?** If yes, it's supervised. If the labels are generated automatically from the data itself, it's self-supervised. If training finds structure without using target labels, it's unsupervised. If the system learns from trial-and-error rewards instead of labels, it's reinforcement. Second, **only for supervised cases:** is the answer being predicted a category (classification) or a number (regression)?

Scenario 1 has known sale prices attached to past houses — that's a label, so supervised; and a price is a number, so regression. Scenario 2 has no labels of any kind — unsupervised. Scenario 3 learns purely from rewards for coverage — reinforcement. Scenario 4 has emails already labelled spam/not-spam — supervised, and the answer is a category, so classification.

### Put It in Order: Counting Each Roll

```python
def count_each(rolls):
    counts = {}
    for r in rolls:
        if r in counts:
            counts[r] = counts[r] + 1
        else:
            counts[r] = 1
    return counts
```

**The order:** B, E, G, I, A, D, H, C — leaving out F.

**How it works:** the `def` line always comes first. The empty dictionary has to be created *before* the loop, so it exists when the loop starts using it. Inside the loop, for each roll `r`, the `if` asks whether `r` has been seen before: if so, add one to its count; `else`, give it a count of 1. The `return` is indented one level inside the function, aligned with the `for` line, so it runs once, after the loop has gone through every roll.

**The line that doesn't belong (F):** `counts = {}` indented *inside* the loop. It would empty the dictionary at the start of every time round, so the function would only ever remember the very last roll. A variable that should build up across a loop must be created once, before the loop starts.

### The Naive Predictor

```python
def predict_next_roll(previous_rolls):
    """A very naive 'model': always predict the most common past roll."""
    return Counter(previous_rolls).most_common(1)[0][0]
```

**How it works:** this is Steps 1 to 4 of the Counter Run & Read, joined into one line. `Counter(previous_rolls)` tallies how often each value appeared. `.most_common(1)` asks for the single most frequent one, returned as a list holding one `(value, count)` pair, e.g. `[(4, 12)]`. `[0]` pulls out that pair, and the second `[0]` pulls the value out of the pair — leaving just `4`.

**A common mistake:** returning the count instead of the value (writing `[0][1]`, or stopping at `[0]`). If your first attempt guessed something like `12`, or `(4, 12)`, instead of a number from 1 to 6, that's exactly what happened.

**Another way, using your `count_each()` function:**

```python
def predict_next_roll(previous_rolls):
    counts = count_each(previous_rolls)
    best_value, best_count = None, 0
    for value, count in counts.items():     # .items() gives each (key, value) pair of the dictionary
        if count > best_count:              # a new "most common so far"?
            best_value, best_count = value, count
    return best_value
```

This keeps track of the best value seen so far and replaces it whenever a bigger count turns up. It's longer, but it shows what `Counter.most_common()` is doing for you behind the scenes.

### Compare Two Baseline Predictors *(stretch)*

```python
def predict_next_roll_v2(previous_rolls):
    """Variant: predict the SECOND most common past roll."""
    counts = Counter(previous_rolls)
    return counts.most_common(2)[1][0]
```

**How it works:** almost identical to `predict_next_roll`, but `.most_common(2)` now asks for the top *two* most frequent values, giving a list of two `(value, count)` pairs. `[1]` selects the *second* one (remember, counting starts at `0`, so `[0]` would be the first/most-common one again), and the final `[0]` pulls the value out of that pair.

### Put It in Order: The Dot Product

```python
def dot_product(v1, v2):
    total = 0
    for a, b in zip(v1, v2):
        total = total + a * b
    return total
```

**The order:** D, F, A, E, C — leaving out B.

**How it works:** it has exactly the same shape as `l1_distance()`: start a running total at zero before the loop, add something for each pair of positions inside the loop, and return the total after the loop. The only difference is *what* gets added: the product `a * b` instead of the distance `abs(a - b)`.

**The line that doesn't belong (B):** `total = a * b` *replaces* the total each time round instead of adding to it, so the function would return only the last product (8 for A and B, instead of 24). Watch for `total = total + ...` whenever you're adding things up in a loop.

### A Vector's Magnitude

```python
def magnitude(v):
    """The length of vector v: square each value, add them up, square-root once at the end."""
    total = 0
    for x in v:
        total = total + x ** 2
    return math.sqrt(total)
```

**How it works:** the same "square, add up, one square root at the end" as `l2_distance()`, except there's only one vector, so the loop goes through its values directly (`for x in v:`) and squares each value `x` itself, instead of a difference `(a - b)`.

**Common mistakes:**
- Adding `math.sqrt(x ** 2)` on each iteration gives absolute values. For `[3, 4]`, their sum is 7, rather than the magnitude `math.sqrt(3 ** 2 + 4 ** 2)`, which is 5.
- Forgetting the `return` line, so the function gives back `None` and the test cell crashes when it tries to round it.

### Rank by Similarity *(stretch)*

```python
similarities = {name: cosine_similarity(reference, seq) for name, seq in candidates.items()}
ranked = sorted(similarities.items(), key=lambda pair: pair[1], reverse=True)
```

**How it works:** the dictionary comprehension computes one similarity score per candidate, giving a dictionary like `{"A": 0.9, "B": 0.4, "C": 0.7}`. `sorted(similarities.items(), ...)` then arranges those `(name, score)` pairs in order; `key=lambda pair: pair[1]` tells `sorted()` to order them by the score (the second item of each pair) rather than the name; `reverse=True` puts the highest score first instead of `sorted()`'s usual smallest-first order.

**The same thing with a plain loop,** if the comprehension feels too compact:

```python
similarities = {}
for name, seq in candidates.items():
    similarities[name] = cosine_similarity(reference, seq)
```

### Put It in Order: A Die With Any Number of Sides

```python
def roll_any_die(n_rolls, sides, seed=None):
    if seed is not None:
        random.seed(seed)
    rolls = []
    for _ in range(n_rolls):
        rolls.append(random.randint(1, sides))
    return rolls
```

**The order:** E, H, B, F, A, G, C — leaving out D. Putting F (`rolls = []`) before H and B also works: both orders set the seed before any random number is made, which is the only thing that matters.

**How it works:** this is `roll_dice_long()` from Step 1, with one extra parameter: `sides` replaces the fixed `6` in `random.randint()`. The seed must be set **before the loop**, because a seed only affects random numbers generated *after* it's set.

**The line that doesn't belong (D):** `rolls = random.randint(1, sides)` replaces the whole list with a single number each time round. To add one item to the end of a list, use `rolls.append(...)`.

**If the checker said the two runs didn't match,** you'd most likely put the seed lines after the loop. The first run then rolled *before* any seed was set, so it couldn't be repeated.

### Build Your Own Reproducible Generator

```python
def generate_measurements(n, low, high, seed=None):
    """Generate n random floats between low and high, reproducible via seed."""
    if seed is not None:
        random.seed(seed)
    return [random.uniform(low, high) for _ in range(n)]
```

**How it works:** this follows exactly the same shape as `roll_any_die()` — check whether a seed was given (`if seed is not None:`), and if so, set it (`random.seed(seed)`) *before* generating anything. Then build the list, just swapping `random.randint(1, sides)` for `random.uniform(low, high)`. The solution above builds the list with a list comprehension; the loop version works identically:

```python
def generate_measurements(n, low, high, seed=None):
    if seed is not None:
        random.seed(seed)
    values = []
    for _ in range(n):
        values.append(random.uniform(low, high))
    return values
```

**A common mistake:** calling `random.seed(seed)` *after* generating the values instead of before. A seed only affects numbers generated *after* it's set — setting it too late has no effect on reproducibility at all, and the two runs won't match.

### Loss and metrics
A loss supplies an error measure to minimise in training; an evaluation metric reports how well predictions meet a goal. Both can summarise many predictions, and the same calculation may serve both roles.

In Lesson 1.2, one notebook costs $3, so two examples are 1 notebook → $3 and 2 notebooks → $6. Predicting with multiplier 2 gives $2 and $4: absolute errors 1 and 2, mean 1.5. Multiplier 3 gives $3 and $6: mean error 0. Choosing the lower-loss multiplier illustrates optimisation. On new labelled examples, mean absolute error can also be an evaluation metric. These two exact training examples do not prove performance on new data.

**Check your paragraph:** did you distinguish purpose rather than single versus many predictions, and explain that two different formulas are not always needed?